In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
%matplotlib inline 
import seaborn as snsw
import gc 
import shap
import re
import xgboost as xgb
import joblib

random_state=14

pd.set_option("display.max_columns",1000)
pd.set_option("display.max_rows",1000)
pd.options.display.float_format = '{:,.3f}'.format

from collections import defaultdict

from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 90%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))

In [ ]:
def gbm_data_prep_noagg(data,features,numerator,denominator,weight,offset):
    if denominator == weight:
        facts = [numerator, denominator]
    else:
        facts = [numerator, denominator, weight]
        
    agg_dict = {i: 'sum' for i in facts}
    
    #data1 = data.groupby(features, dropna = False).agg(agg_dict).reset_index()
    data1 = data.fillna(0)
    data1['target'] = data1[numerator].fillna(0)/data1[denominator]
    
    data1 = data1[(features + ['target'] + facts + [offset])]
    
    print(data1[weight] )

    data1 = data1.loc[data1[weight] > 0]
    data1 = data1.loc[data1[denominator] > 0]
    data1 = data1.loc[data1[numerator] >= 0]
    
    return data1

def gbm_data_prep(data,features,numerator,denominator,weight):
    if denominator == weight:
        facts = [numerator, denominator]
    else:
        facts = [numerator, denominator, weight]
        
    agg_dict = {i: 'sum' for i in facts}
    
    data1 = data.groupby(features, dropna = False).agg(agg_dict).reset_index()
    data1 = data1.fillna(0)
    data1['target'] = data1[numerator].fillna(0)/data1[denominator]
    
    data1 = data1[(features + ['target'] + facts + [offset])]
    
    data1 = data1.loc[data1[weight] > 0]
    data1 = data1.loc[data1[denominator] > 0]
    data1 = data1.loc[data1[numerator] >= 0]
    
    return data1

def xgb_mono(x):
        return str(x).replace('[','(').replace(']',')')

def xbg_wrapper(train,test,passby,numerator,denominator,weight,offset,monotone_constraint,tweedie_p = 1.5,bmf = .10,max_depth = 6,num_round = 50,min_child_weight = 1,subsample = 1,colsample_bytree = 1,alph = 0):
    d_train = gbm_data_prep_noagg(train,features,numerator,denominator,weight,offset)
    d_test = gbm_data_prep_noagg(test,features,numerator,denominator,weight,offset)
    
    params = {
        'device' : 'cuda',
        'objective': 'reg:tweedie',
        'tweedie_variance_power': tweedie_p,
        'learning_rate': bmf,
        'verbosity': 1,
        'max_depth': max_depth,
        'min_child_weight': min_child_weight,
        'subsample': subsample,
        'colsample_bytree': colsample_bytree,
        'alpha': alph,
        'base_score': d_train[numerator].sum()/d_train[numerator].sum(),
        #'base_score': d_train[numerator].sum(),
        'monotone_constraints': xgb_mono(monotone_constraint),
        'tree_method': 'hist'
        #'predictor': 'gpu_predictor'
        }
    
    weight_train = d_train['weight']
    dtrain = xgb.DMatrix(d_train.drop(columns = [numerator,denominator,offset,passby] + ['target','weight']), label=d_train['target'], weight=weight_train)
    
    num_round = num_round
    global model_xgb
    model_xgb = xgb.train(params, dtrain, num_round)
    
    weight_test = d_test['weight']
    dtest = xgb.DMatrix(d_test.drop(columns = [numerator,denominator,offset,passby] + ['target','weight']), label=d_test['target'])
    ypred_test = model_xgb.predict(dtest)
    
    ypred_train = model_xgb.predict(dtrain)

    def apply_predictions():
        global out_train
        global out_test
        
        out_train = d_train.copy()
        out_train['pred'] = ypred_train
        out_train['incurred_act'] = out_train[numerator]
        out_train['denom'] = out_train[denominator]
        out_train['incurred_pred'] = out_train['denom'] * out_train['pred']
        out_train['incurred_act_offsetadded'] = out_train[numerator]*out_train['offset']
        out_train['incurred_pred_offsetadded'] = out_train['denom'] *out_train['pred']* out_train['offset']
        out_train['denom_offsetadded'] = out_train[denominator]* out_train['offset']
                       
        out_test = d_test.copy()
        out_test['pred'] = ypred_test
        out_test['incurred_act'] = out_test[numerator]
        out_test['denom'] = out_test[denominator]
        out_test['incurred_pred'] = out_test['denom'] * out_test['pred']
        out_test['incurred_act_offsetadded'] = out_test[numerator]*out_test['offset']
        out_test['incurred_pred_offsetadded'] = out_test['denom'] * out_test['pred']*out_test['offset']
        out_test['denom_offsetadded'] = out_test[denominator]*out_test['offset']
        
        
    apply_predictions()
    
def rnd(data,var,level):
    data[var] = round(data[var]/level,0)*level
    
def box_var(data,var,low,high):
    data[var] = np.where(data[var] > high, high, data[var])
    data[var] = np.where(data[var] < low, low, data[var])

def lift_chart_withoffset(test_data, weight_name, bins, print_table = False):
    test_data['decile'] = (round(test_data.sort_values(by = 'pred')[weight_name].cumsum()/test_data[weight_name].sum(),2)*bins).apply(np.floor)
    test_data['decile'] = np.where(test_data['decile'] + 1 > bins ,bins,test_data['decile'] + 1)
    
    x = test_data.groupby(['decile'], dropna = False).agg({weight_name: 'sum', 'incurred_act_offsetadded': 'sum', 'incurred_pred_offsetadded': 'sum', 'denom_offsetadded': 'sum'}).reset_index()
    
    x['act'] = x['incurred_act_offsetadded']/x['denom_offsetadded']
    x['pred'] = x['incurred_pred_offsetadded']/x['denom_offsetadded']
    x.drop(columns = ['incurred_act_offsetadded','incurred_pred_offsetadded','denom_offsetadded'], inplace = True)
    
    dfg = x
    fig, ax = plt.subplots(figsize=(12,6))
    ax2  = ax.twinx()
    
    y_max = np.where(dfg['act'].max() > dfg['pred'].max(),dfg['act'].max(),dfg['pred'].max())*1.20
    ax2.set_ylim(0,y_max)
    
    dfg[weight_name].plot.bar(stacked=False, ax=ax, alpha=0.6)
    dfg['act'].plot(kind='line', ax=ax2, marker='o', linewidth = 0, legend='act')
    dfg['pred'].plot(kind='line', ax=ax2, marker='o', legend='pred')
    plt.show()
    
    if print_table == True:
        print(x)


def lift_chart(test_data, weight_name, bins, print_table = False):
    test_data['decile'] = (round(test_data.sort_values(by = 'pred')[weight_name].cumsum()/test_data[weight_name].sum(),2)*bins).apply(np.floor)
    test_data['decile'] = np.where(test_data['decile'] + 1 > bins ,bins,test_data['decile'] + 1)
    x = test_data.groupby(['decile'], dropna = False).agg({weight_name: 'sum', 'incurred_act': 'sum', 'incurred_pred': 'sum', 'denom': 'sum'}).reset_index()
    
    x['act'] = x['incurred_act']/x['denom']
    x['pred'] = x['incurred_pred']/x['denom']
    x.drop(columns = ['incurred_act','incurred_pred','denom'], inplace = True)
    
    dfg = x
    fig, ax = plt.subplots(figsize=(12,6))
    ax2  = ax.twinx()
    
    y_max = np.where(dfg['act'].max() > dfg['pred'].max(),dfg['act'].max(),dfg['pred'].max())*1.20
    ax2.set_ylim(0,y_max)
    
    dfg[weight_name].plot.bar(stacked=False, ax=ax, alpha=0.6)
    dfg['act'].plot(kind='line', ax=ax2, marker='o', linewidth = 0, legend='act')
    dfg['pred'].plot(kind='line', ax=ax2, marker='o', legend='pred')
    plt.show()
    
    if print_table == True:
        print(x)
    
def resid_plot(data,feature,weight, print_table = False, round_value = 2):
    agg_dict = {weight: 'sum', 'incurred_act': 'sum', 'incurred_pred': 'sum', 'denom': 'sum'}
    x = data.groupby([feature]).agg(agg_dict).reset_index()
    x[feature] = round(x[feature],round_value)
    
    if x.shape[0] > 50:
        x = x.groupby(pd.qcut(x[feature], q = 20, duplicates = 'drop')).agg(agg_dict).reset_index()
        
    x['act'] = x['incurred_act']/x['denom']
    x['pred'] = x['incurred_pred']/x['denom']

    fig, ax = plt.subplots(figsize=(12,6))
    ax2  = ax.twinx()
    
    y_max = np.where(x['act'].max() > x['pred'].max(),x['act'].max(),x['pred'].max())*1.20
    ax2.set_ylim(0,y_max)

    x[weight].plot.bar(stacked=False, ax=ax, alpha=0.6)
    x['act'].plot(kind='line', ax=ax2, marker='o', legend='act')
    x['pred'].plot(kind='line', ax=ax2, marker='o', legend='pred')
    
    plt.xticks(ticks = x.index, labels = x[feature])

    ax.set(ylabel=weight, title = feature)
    plt.show()
    
    if print_table == True:
        print(x[([feature, weight, 'pred'])])

def test_gini():
    actual = [1,2,4,8]
    pred = [1,2,4,8]
    print(gini(actual, pred))
    
def gini(actual, pred):
    actual = np.array(actual)
    pred = np.array(pred)
    
    # Sort by the predicted values (descending order)
    sorted_indices = np.argsort(pred)
    sorted_actual = actual[sorted_indices]
    
    # Cumulative sums of actual values and population share
    cumulative_actual = np.cumsum(sorted_actual) / np.sum(sorted_actual)
    cumulative_population = np.linspace(1/len(sorted_actual), 1, len(sorted_actual))
    
    # Area under the Lorenz curve using trapezoidal integration
    area_under_curve = np.trapz(cumulative_actual, cumulative_population)
    
    # Gini is 1 - 2 * area under Lorenz curve
    gini_value = 1 - 2 * area_under_curve
    
    return gini_value

def plot_lorenz_curve(actual, pred, title="Lorenz Curve"):
    actual = np.array(actual)
    pred = np.array(pred)
    
    sorted_indices = np.argsort(pred) #(descending for compatibility with your gini logic)
    sorted_actual = actual[sorted_indices]
    
    cumulative_actual = np.cumsum(sorted_actual) / np.sum(sorted_actual)
    cumulative_population = np.linspace(1/len(sorted_actual), 1, len(sorted_actual))
    
    cumulative_actual = np.insert(cumulative_actual, 0, 0)
    cumulative_population = np.insert(cumulative_population, 0, 0)

    #gini_value = 1 - 2 * np.trapz(cumulative_actual, cumulative_population)
    gini_value = gini(actual, pred)
    
    # Plot
    plt.figure(figsize=(8, 6))
    plt.plot(cumulative_population, cumulative_actual, label='Lorenz Curve', color='blue')
    plt.plot([0, 1], [0, 1], label='Line of Equality', color='black', linestyle='--')
    plt.fill_between(cumulative_population, cumulative_actual, cumulative_population, color='gray', alpha=0.3)
    plt.title(f"{title}\nGini Coefficient = {gini_value:.4f}")
    plt.xlabel("Cumulative Share of Population")
    plt.ylabel("Cumulative Share of Actuals")
    plt.legend()
    plt.grid(True)
    plt.show()
    
def lift_chart_2025update(test_data, weight_name, bins, print_table=False):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # Calculate decile groups
    test_data['decile'] = (round(test_data.sort_values(by='pred')[weight_name].cumsum() / test_data[weight_name].sum(), 2) * bins).apply(np.floor)
    test_data['decile'] = np.where(test_data['decile'] + 1 > bins, bins, test_data['decile'] + 1)
    
    # Group data by deciles
    x = test_data.groupby(['decile'], dropna=False).agg({weight_name: 'sum', 'incurred_act': 'sum', 'incurred_pred': 'sum', 'denom': 'sum'}).reset_index()
    x['act'] = x['incurred_act'] / x['denom']
    x['pred'] = x['incurred_pred'] / x['denom']
    x.drop(columns=['incurred_act', 'incurred_pred', 'denom'], inplace=True)

    # Prepare data for plotting
    dfg = x
    fig, ax = plt.subplots(figsize=(12, 6))
    ax2 = ax.twinx()

    # Set limits for axes
    y_max = max(dfg['act'].max(), dfg['pred'].max()) * 1.20
    ax.set_ylim(0, y_max)  # Left y-axis (act and pred)
    ax2.set_ylim(0, 100)   # Right y-axis (weights as percentage)

    # Plot bar for weights
    (dfg[weight_name] / dfg[weight_name].sum() * 100).plot.bar(stacked=False, ax=ax2, alpha=0.6)

    # Plot lines for act and pred
    dfg['act'].plot(kind='line', ax=ax, marker='o', linewidth=1, color='blue', label='act')
    dfg['pred'].plot(kind='line', ax=ax, marker='o', linewidth=1, color='orange', label='pred')

    # Add legends and labels
    ax.set_ylabel("Values (act and pred)")
    ax2.set_ylabel("Weights (%)")
    ax.set_xlabel("Decile")
    ax.legend(loc='upper left')
    ax2.legend(["Weights"], loc='upper right')

    plt.show()

    if print_table:
        print(x)
        print('______________________________')
    gini_gbm = gini(test_data['target'], test_data['pred'])
    print(f" Gini Coefficient from actual: {gini_gbm}")
    plot_lorenz_curve(test_data['target'], test_data['pred'], title="Lorenz Curve: Raw Predictions")

    print('______________________________')
    gini_gbm = gini(x['act'], x['pred'])
    print(f" Gini Coefficient from lift: {gini_gbm}")
    plot_lorenz_curve(x['act'], x['pred'], title="Lorenz Curve: Lift-Aggregated Predictions")


In [ ]:
def lift_chart_2026update(test_data, weight_name, bins, print_table=False, relativity_base="pred"):
    """
    Actuarial-style lift chart using relativities (decile value / overall mean).

    Parameters
    ----------
    test_data : pd.DataFrame
        Must contain: 'pred', 'incurred_act', 'incurred_pred', 'denom' (and optionally 'target').
    weight_name : str
        Column name for exposure/weights used to create weighted deciles and weight bars.
    bins : int
        Number of bins (typically 10 for deciles).
    print_table : bool
        If True, prints the decile table.
    relativity_base : str
        'pred' -> pred_rel = pred / overall_pred  (shape-only comparison; removes calibration)
        'act'  -> pred_rel = pred / overall_act   (shows calibration bias vs actual baseline)

    Notes
    -----
    - This does NOT change model ranking; it only normalizes scale to make comparisons easier.
    - If denom is all 1, then act/pred are just decile averages; still consistent.
    """

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # --- Defensive copy + checks ---
    df = test_data.copy()

    required = {'pred', 'incurred_act', 'incurred_pred', 'denom', weight_name}
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if bins <= 1:
        raise ValueError("bins must be > 1")

    # Avoid division by zero in denom
    if (df['denom'] <= 0).any():
        raise ValueError("denom must be > 0 for all rows (or pre-filter / fix zeros).")

    # --- Weighted deciles based on pred ---
    df = df.sort_values('pred').reset_index(drop=True)
    w = df[weight_name].astype(float).fillna(0.0)
    wsum = w.sum()
    if wsum <= 0:
        raise ValueError(f"Sum of {weight_name} must be > 0")

    cum_w = (w.cumsum() / wsum)
    df['decile'] = np.ceil(cum_w * bins).astype(int)
    df['decile'] = df['decile'].clip(1, bins)

    # --- Aggregate by decile ---
    x = (
        df.groupby('decile', dropna=False)
          .agg(**{
              weight_name: (weight_name, 'sum'),
              'incurred_act_sum': ('incurred_act', 'sum'),
              'incurred_pred_sum': ('incurred_pred', 'sum'),
              'denom_sum': ('denom', 'sum'),
          })
          .reset_index()
          .sort_values('decile')
    )

    # Raw (level) decile values
    x['act'] = x['incurred_act_sum'] / x['denom_sum']
    x['pred'] = x['incurred_pred_sum'] / x['denom_sum']

    # --- Overall (portfolio) levels (same formula) ---
    overall_act = df['incurred_act'].sum() / df['denom'].sum()
    overall_pred = df['incurred_pred'].sum() / df['denom'].sum()

    # --- Relativities (actuarial style) ---
    x['act_rel'] = x['act'] / overall_act

    if relativity_base not in ("pred", "act"):
        raise ValueError("relativity_base must be 'pred' or 'act'")

    # pred relativity normalized either to its own mean (pred) or to actual mean (act)
    if relativity_base == "pred":
        x['pred_rel'] = x['pred'] / overall_pred
    else:
        x['pred_rel'] = x['pred'] / overall_act

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(12, 6))
    ax2 = ax.twinx()

    # Weight bars (%)
    weight_pct = (x[weight_name] / x[weight_name].sum() * 100.0)
    weight_pct.plot.bar(stacked=False, ax=ax2, alpha=0.6)

    # Relativity lines
    x['act_rel'].plot(kind='line', ax=ax, marker='o', linewidth=1, label='act relativity')
    x['pred_rel'].plot(kind='line', ax=ax, marker='o', linewidth=1, label='pred relativity')

    # Y-axis: relativity window around your data (not starting at 0)
    y_max = max(x['act_rel'].max(), x['pred_rel'].max()) * 1.10
    y_min = min(x['act_rel'].min(), x['pred_rel'].min()) * 0.90

    # Keep it sane if extremely tight range
    if np.isclose(y_max, y_min):
        y_min, y_max = 0.95, 1.05

    ax.set_ylim(y_min, y_max)
    ax2.set_ylim(0, 100)

    # Baseline at 1.0 (portfolio average)
    ax.axhline(1.0, linewidth=1, linestyle='--')

    # Labels / legends
    ax.set_ylabel("Relativity (vs portfolio average)")
    ax2.set_ylabel("Weights (%)")
    ax.set_xlabel("Decile")

    ax.legend(loc='upper left')
    ax2.legend(["Weights"], loc='upper right')

    plt.show()

    # --- Optional table ---
    if print_table:
        out_cols = ['decile', weight_name, 'act', 'pred', 'act_rel', 'pred_rel']
        print(x[out_cols])
        print('______________________________')

    # --- Your existing Gini/Lorenz calls (only if available) ---
    # Note: these depend on your external gini() and plot_lorenz_curve() functions.
    if 'target' in df.columns:
        try:
            gini_gbm = gini(df['target'], df['pred'])
            print(f" Gini Coefficient from actual: {gini_gbm}")
            plot_lorenz_curve(df['target'], df['pred'], title="Lorenz Curve: Raw Predictions")
        except NameError:
            pass

    try:
        print('______________________________')
        gini_gbm = gini(x['act'], x['pred'])
        print(f" Gini Coefficient from lift: {gini_gbm}")
        plot_lorenz_curve(x['act'], x['pred'], title="Lorenz Curve: Lift-Aggregated Predictions")
    except NameError:
        pass

    return x

In [ ]:
def lift_chart_2026update_earned_premium(
    test_data,
    weight_name,
    bins,
    print_table=False,
    relativity_base="act",
    earned_premium_col="earned_premium"
):
    """
    Actuarial-style lift chart using relativities with earned premium as denominator.

    Parameters
    ----------
    test_data : pd.DataFrame
        Must contain:
        - 'pred'               : ranking score used for sorting into lift buckets
        - 'incurred_act'       : actual incurred loss amount
        - 'incurred_pred'      : predicted incurred loss amount
        - earned_premium_col   : earned premium
        - weight_name          : weights used to form weighted deciles
    weight_name : str
        Column used to create weighted deciles and weight bars.
        In many cases this should also be earned premium.
    bins : int
        Number of bins (typically 10 for deciles).
    print_table : bool
        If True, prints the decile table.
    relativity_base : str
        'pred' -> pred_rel = pred / overall_pred
        'act'  -> pred_rel = pred / overall_act
    earned_premium_col : str
        Column name for earned premium.

    Notes
    -----
    - This assumes incurred_act and incurred_pred are loss amounts, not already pure premiums.
    - Decile values are loss cost / pure premium:
          sum(loss) / sum(earned premium)
    - Rows with earned premium <= 0 are dropped because EP-based loss cost is undefined.
    """

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # --- Defensive copy + checks ---
    df = test_data.copy()

    required = {'pred', 'incurred_act', 'incurred_pred', earned_premium_col, weight_name}
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if bins <= 1:
        raise ValueError("bins must be > 1")

    # Remove rows with missing essentials
    df = df.dropna(subset=['pred', 'incurred_act', 'incurred_pred', earned_premium_col, weight_name]).copy()

    # Weights for bucketing must be nonnegative
    if (df[weight_name] < 0).any():
        raise ValueError(f"{weight_name} must be >= 0 for all rows.")

    # --- Filter EP <= 0 rows ---
    bad_ep_mask = df[earned_premium_col] <= 0
    dropped_rows = bad_ep_mask.sum()

    if dropped_rows > 0:
        dropped_actual = df.loc[bad_ep_mask, 'incurred_act'].sum()
        dropped_pred = df.loc[bad_ep_mask, 'incurred_pred'].sum()
        total_actual = df['incurred_act'].sum()
        total_pred = df['incurred_pred'].sum()

        actual_share = dropped_actual / total_actual if total_actual != 0 else np.nan
        pred_share = dropped_pred / total_pred if total_pred != 0 else np.nan

        print(
            f"Dropping {dropped_rows:,} rows where {earned_premium_col} <= 0 "
            f"for EP-based lift chart. "
            f"Excluded actual loss share = {actual_share:.4%}, "
            f"excluded predicted loss share = {pred_share:.4%}"
        )

        df = df.loc[~bad_ep_mask].copy()

    if df.empty:
        raise ValueError("No rows remain after filtering earned premium <= 0.")

    # --- Weighted deciles based on pred ---
    df = df.sort_values('pred').reset_index(drop=True)

    w = df[weight_name].astype(float).fillna(0.0)
    wsum = w.sum()
    if wsum <= 0:
        raise ValueError(f"Sum of {weight_name} must be > 0")

    cum_w = w.cumsum() / wsum
    df['decile'] = np.ceil(cum_w * bins).astype(int).clip(1, bins)

    # --- Aggregate by decile ---
    x = (
        df.groupby('decile', dropna=False)
          .agg(**{
              weight_name: (weight_name, 'sum'),
              'earned_premium_sum': (earned_premium_col, 'sum'),
              'incurred_act_sum': ('incurred_act', 'sum'),
              'incurred_pred_sum': ('incurred_pred', 'sum'),
          })
          .reset_index()
          .sort_values('decile')
    )

    # --- Decile level loss cost / pure premium ---
    x['act'] = x['incurred_act_sum'] / x['earned_premium_sum']
    x['pred'] = x['incurred_pred_sum'] / x['earned_premium_sum']

    # --- Overall portfolio level ---
    overall_act = df['incurred_act'].sum() / df[earned_premium_col].sum()
    overall_pred = df['incurred_pred'].sum() / df[earned_premium_col].sum()

    # --- Relativities ---
    x['act_rel'] = x['act'] / overall_act

    if relativity_base not in ("pred", "act"):
        raise ValueError("relativity_base must be 'pred' or 'act'")

    if relativity_base == "pred":
        x['pred_rel'] = x['pred'] / overall_pred
    else:
        x['pred_rel'] = x['pred'] / overall_act

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(12, 6))
    ax2 = ax.twinx()

    # Weight bars (%)
    weight_pct = x[weight_name] / x[weight_name].sum() * 100.0
    ax2.bar(x['decile'], weight_pct, alpha=0.35, label='Weights')

    # Relativity lines
    ax.plot(x['decile'], x['act_rel'], marker='o', linewidth=1.5, label='act relativity')
    ax.plot(x['decile'], x['pred_rel'], marker='o', linewidth=1.5, label='pred relativity')

    # Y-axis window
    y_max = max(x['act_rel'].max(), x['pred_rel'].max()) * 1.10
    y_min = min(x['act_rel'].min(), x['pred_rel'].min()) * 0.90

    if np.isclose(y_max, y_min):
        y_min, y_max = 0.95, 1.05

    ax.set_ylim(y_min, y_max)
    ax2.set_ylim(0, max(100, weight_pct.max() * 1.2))

    # Portfolio baseline
    ax.axhline(1.0, linewidth=1, linestyle='--')

    # Labels / legends
    ax.set_ylabel("Relativity (vs portfolio average)")
    ax2.set_ylabel("Weights (%)")
    ax.set_xlabel("Decile")
    ax.set_xticks(x['decile'])

    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')

    plt.title("Lift Chart using Earned Premium")
    plt.show()

    # --- Optional table ---
    if print_table:
        out_cols = [
            'decile',
            weight_name,
            'earned_premium_sum',
            'incurred_act_sum',
            'incurred_pred_sum',
            'act',
            'pred',
            'act_rel',
            'pred_rel'
        ]
        print(x[out_cols])
        print('______________________________')

    # --- Optional Gini/Lorenz calls ---
    if 'target' in df.columns:
        try:
            gini_gbm = gini(df['target'], df['pred'])
            print(f"Gini Coefficient from actual: {gini_gbm}")
            plot_lorenz_curve(df['target'], df['pred'], title="Lorenz Curve: Raw Predictions")
        except NameError:
            pass

    try:
        print('______________________________')
        gini_gbm = gini(x['act'], x['pred'])
        print(f"Gini Coefficient from lift: {gini_gbm}")
        plot_lorenz_curve(x['act'], x['pred'], title="Lorenz Curve: Lift-Aggregated Predictions")
    except NameError:
        pass

    return x

In [ ]:
# def lift_chart_2026update_earned_premium(
#     test_data,
#     weight_name,
#     bins,
#     print_table=False,
#     relativity_base="act",
#     earned_premium_col="earned_premium"
# ):
#     """
#     Actuarial-style lift chart using relativities with earned premium as denominator.

#     Parameters
#     ----------
#     test_data : pd.DataFrame
#         Must contain:
#         - 'pred'               : ranking score used for sorting into lift buckets
#         - 'incurred_act'       : actual incurred loss amount
#         - 'incurred_pred'      : predicted incurred loss amount
#         - earned_premium_col   : earned premium
#         - weight_name          : weights used to form weighted deciles
#     weight_name : str
#         Column used to create weighted deciles and weight bars.
#         In many cases this should also be earned premium.
#     bins : int
#         Number of bins (typically 10 for deciles).
#     print_table : bool
#         If True, prints the decile table.
#     relativity_base : str
#         'pred' -> pred_rel = pred / overall_pred
#         'act'  -> pred_rel = pred / overall_act
#     earned_premium_col : str
#         Column name for earned premium.

#     Notes
#     -----
#     - This assumes incurred_act and incurred_pred are loss amounts, not already pure premiums.
#     - Decile values are loss costs / pure premiums:
#           sum(loss) / sum(earned premium)
#     - For direct comparison, 'act' is usually the better relativity base.
#     """

#     import numpy as np
#     import pandas as pd
#     import matplotlib.pyplot as plt

#     # --- Defensive copy + checks ---
#     df = test_data.copy()

#     required = {'pred', 'incurred_act', 'incurred_pred', earned_premium_col, weight_name}
#     missing = [c for c in required if c not in df.columns]
#     if missing:
#         raise ValueError(f"Missing required columns: {missing}")

#     if bins <= 1:
#         raise ValueError("bins must be > 1")

#     # Remove rows with missing essentials
#     df = df.dropna(subset=['pred', 'incurred_act', 'incurred_pred', earned_premium_col, weight_name]).copy()

#     # Earned premium must be positive
#     if (df[earned_premium_col] <= 0).any():
#         raise ValueError(f"{earned_premium_col} must be > 0 for all rows (or pre-filter/fix zeros).")

#     # Weights for bucketing must be nonnegative
#     if (df[weight_name] < 0).any():
#         raise ValueError(f"{weight_name} must be >= 0 for all rows.")

#     # --- Weighted deciles based on pred ---
#     df = df.sort_values('pred').reset_index(drop=True)

#     w = df[weight_name].astype(float).fillna(0.0)
#     wsum = w.sum()
#     if wsum <= 0:
#         raise ValueError(f"Sum of {weight_name} must be > 0")

#     cum_w = w.cumsum() / wsum
#     df['decile'] = np.ceil(cum_w * bins).astype(int).clip(1, bins)

#     # --- Aggregate by decile ---
#     x = (
#         df.groupby('decile', dropna=False)
#           .agg(**{
#               weight_name: (weight_name, 'sum'),
#               'earned_premium_sum': (earned_premium_col, 'sum'),
#               'incurred_act_sum': ('incurred_act', 'sum'),
#               'incurred_pred_sum': ('incurred_pred', 'sum'),
#           })
#           .reset_index()
#           .sort_values('decile')
#     )

#     # --- Decile level loss cost / pure premium ---
#     x['act'] = x['incurred_act_sum'] / x['earned_premium_sum']
#     x['pred'] = x['incurred_pred_sum'] / x['earned_premium_sum']

#     # --- Overall portfolio level ---
#     overall_act = df['incurred_act'].sum() / df[earned_premium_col].sum()
#     overall_pred = df['incurred_pred'].sum() / df[earned_premium_col].sum()

#     # --- Relativities ---
#     x['act_rel'] = x['act'] / overall_act

#     if relativity_base not in ("pred", "act"):
#         raise ValueError("relativity_base must be 'pred' or 'act'")

#     if relativity_base == "pred":
#         x['pred_rel'] = x['pred'] / overall_pred
#     else:
#         x['pred_rel'] = x['pred'] / overall_act

#     # --- Plot ---
#     fig, ax = plt.subplots(figsize=(12, 6))
#     ax2 = ax.twinx()

#     # Weight bars (%)
#     weight_pct = x[weight_name] / x[weight_name].sum() * 100.0
#     ax2.bar(x['decile'], weight_pct, alpha=0.35, label='Weights')

#     # Relativity lines
#     ax.plot(x['decile'], x['act_rel'], marker='o', linewidth=1.5, label='act relativity')
#     ax.plot(x['decile'], x['pred_rel'], marker='o', linewidth=1.5, label='pred relativity')

#     # Y-axis window
#     y_max = max(x['act_rel'].max(), x['pred_rel'].max()) * 1.10
#     y_min = min(x['act_rel'].min(), x['pred_rel'].min()) * 0.90

#     if np.isclose(y_max, y_min):
#         y_min, y_max = 0.95, 1.05

#     ax.set_ylim(y_min, y_max)
#     ax2.set_ylim(0, max(100, weight_pct.max() * 1.2))

#     # Portfolio baseline
#     ax.axhline(1.0, linewidth=1, linestyle='--')

#     # Labels / legends
#     ax.set_ylabel("Relativity (vs portfolio average)")
#     ax2.set_ylabel("Weights (%)")
#     ax.set_xlabel("Decile")
#     ax.set_xticks(x['decile'])

#     ax.legend(loc='upper left')
#     ax2.legend(loc='upper right')

#     plt.title("Lift Chart using Earned Premium")
#     plt.show()

#     # --- Optional table ---
#     if print_table:
#         out_cols = [
#             'decile',
#             weight_name,
#             'earned_premium_sum',
#             'incurred_act_sum',
#             'incurred_pred_sum',
#             'act',
#             'pred',
#             'act_rel',
#             'pred_rel'
#         ]
#         print(x[out_cols])
#         print('______________________________')

#     # --- Optional Gini/Lorenz calls ---
#     if 'target' in df.columns:
#         try:
#             gini_gbm = gini(df['target'], df['pred'])
#             print(f"Gini Coefficient from actual: {gini_gbm}")
#             plot_lorenz_curve(df['target'], df['pred'], title="Lorenz Curve: Raw Predictions")
#         except NameError:
#             pass

#     try:
#         print('______________________________')
#         gini_gbm = gini(x['act'], x['pred'])
#         print(f"Gini Coefficient from lift: {gini_gbm}")
#         plot_lorenz_curve(x['act'], x['pred'], title="Lorenz Curve: Lift-Aggregated Predictions")
#     except NameError:
#         pass

#     return x

In [ ]:
def shap_bar_pct(shap_values, max_display=40):
    
    # get shap matrix
    values = shap_values.values
    feature_names = shap_values.feature_names
    
    # mean(|shap|)
    importance = np.abs(values).mean(axis=0)
    
    # convert to percentage
    pct_importance = importance / importance.sum() * 100
    
    df = pd.DataFrame({
        "feature": feature_names,
        "pct_importance": pct_importance
    })
    
    # sort
    df = df.sort_values("pct_importance", ascending=False)
    
    # take top features
    top = df.head(max_display)
    
    # compute remaining importance
    remaining = df.iloc[max_display:]["pct_importance"].sum()
    
    if remaining > 0:
        top = pd.concat([
            top,
            pd.DataFrame({
                "feature": [f"Sum of {len(df) - max_display} other features"],
                "pct_importance": [remaining]
            })
        ])
    
    # reverse for barh plotting
    top = top[::-1]
    
    # plot
    plt.figure(figsize=(8,10))
    
    bars = plt.barh(top["feature"], top["pct_importance"], color="#1f77b4")
    
    for i, v in enumerate(top["pct_importance"]):
        plt.text(v + 0.1, i, f"{v:.1f}%", va="center")
    
    plt.xlabel("Percentage Contribution to Model")
    plt.title("SHAP Feature Importance (%)")
    
    plt.tight_layout()
    plt.show()

In [ ]:
import seaborn as sns

def shap_ridge_bool_table(field,digits):
    val_df = df0[([field])].copy()
    val_df.rename(columns = {field: 'val'}, inplace = True)

    val_df['shap'] = fc_df[field]
    val_df['shap'] = round(val_df['shap'],digits)
    val_df['weight'] = fc_df['weight']

    x = val_df.groupby(['val','shap']).agg({'weight': 'sum'}).reset_index()
    
    x['field'] = field
    x = x[(['field','val','shap','weight'])]
    return x

def shap_display(field,data,digits):
    # starting table to merge to
    agg1 = data.groupby(['field','val']).agg({'weight': 'sum'}).reset_index()
    agg1['pct_weight'] = agg1['weight']/agg1['weight'].sum()
    
    agg2 = data.loc[data['shap'] != 0].groupby(['field','val']).agg({'weight': 'sum'}).reset_index()
    agg2.rename(columns = {'weight': 'weight_used'}, inplace = True)

    agg3 = agg1.merge(agg2)
    agg3['pct_used'] = agg3['weight_used']/agg3['weight']

    del agg3['weight'], agg3['weight_used']
    
    # setup for calculation of percentiles and mean
    data2 = data.loc[data['shap'] != 0].copy()
    data2['cumsum_weight'] = data2.groupby(['field','val'])['weight'].cumsum()

    cumsum_max = data2.groupby(['field','val']).agg({'weight': 'sum'}).reset_index()
    cumsum_max.rename(columns = {'weight': 'tot_weight'}, inplace = True)

    data2 = data2.merge(cumsum_max)

    data2['cum_pct'] = data2['cumsum_weight']/data2['tot_weight']
    
    # calculate percentiles and mean
    ## mean
    data2['sp'] = data2['shap'] * data2['weight']
    agg_mean = data2.groupby(['field','val']).agg({'sp': 'sum', 'weight': 'sum'}).reset_index()
    agg_mean['shap_mean'] = agg_mean['sp']/agg_mean['weight']
    agg_mean.drop(columns = ['sp','weight'], inplace = True)
    
    def shap_ntile(data2,ntile,digits):
        data2['p95_diff'] = data2['cum_pct'] - ntile/100
        data2['p95_diff_sign'] = np.sign(data2['p95_diff'])
        data2['p95_diff_abs'] = np.abs(data2['p95_diff'])
        sign_min95 = data2.groupby(['field','val','p95_diff_sign']).agg({'p95_diff_abs': 'min'}).reset_index()
        sign_min95 = sign_min95.merge(data2[(['field','val','p95_diff_sign','p95_diff_abs','shap','p95_diff'])])

        within_val_min95 = sign_min95.groupby(['field','val']).agg({'p95_diff': 'min'}).reset_index()
        within_val_min95.rename(columns = {'p95_diff': 'p95_diff_min'}, inplace = True)
        within_val_max95 = sign_min95.groupby(['field','val']).agg({'p95_diff': 'max'}).reset_index()
        within_val_max95.rename(columns = {'p95_diff': 'p95_diff_max'}, inplace = True)

        sign_min95 = sign_min95.merge(within_val_min95).merge(within_val_max95)
        sign_min95['tot_space'] = sign_min95['p95_diff_max'] - sign_min95['p95_diff_min']
        sign_min95['neg_weight'] = (0 - sign_min95['p95_diff_min'])/sign_min95['tot_space']
        sign_min95['pos_weight'] = (sign_min95['p95_diff_max'])/sign_min95['tot_space']

        shap_min = sign_min95.groupby(['field','val']).agg({'shap': 'min'}).reset_index()
        shap_min.rename(columns = {'shap': 'shap_min'}, inplace = True)
        shap_max = sign_min95.groupby(['field','val']).agg({'shap': 'max'}).reset_index()
        shap_max.rename(columns = {'shap': 'shap_max'}, inplace = True)

        sign_min95 = sign_min95.groupby(['field','val']).agg({'neg_weight': 'mean','pos_weight': 'mean'}).reset_index().merge(shap_min).merge(shap_max)

        sign_min95['shap_p' + str(ntile)] = np.where(sign_min95['pos_weight'] == np.inf, 
                                        sign_min95['shap_max'], 
                                        round(sign_min95['neg_weight'] * sign_min95['shap_min'] + sign_min95['pos_weight'] * sign_min95['shap_max'] ,digits)
                                       )
        df_p95 = sign_min95[(['field','val','shap_p' + str(ntile)])].copy()

        return df_p95
    
    p0 = shap_ntile(data2,0,digits)
    p5 = shap_ntile(data2,5,digits)
    p95 = shap_ntile(data2,95,digits)
    p100 = shap_ntile(data2,100,digits)
    
    agg4 = agg3.merge(p0).merge(p5).merge(agg_mean).merge(p95).merge(p100)
    
    return agg4

def shap_summary_display(train,weight,non_bools,sort_by_shap = False):
    global df0
    df0 = train[(model_xgb.feature_names)].copy().reset_index().drop(columns = 'index')
    df = xgb.DMatrix(df0)
    fc = model_xgb.predict(df, pred_contribs=True)
    global fc_df
    fc_df = pd.DataFrame(fc, columns = model_xgb.feature_names + ['bias'])
    del fc_df['bias']
    fc_df['weight'] = train[weight]

    for i in non_bools:
        try:
            del fc_df[i]
        except:
            pass

    global result
    result = pd.DataFrame()
    for field in [i for i in fc_df.columns if i != 'weight']:
        display = shap_display(field,shap_ridge_bool_table(field,5),5)
        result = pd.concat([result,display])

    result['tot_wtd_shap_abs'] = np.abs(result['pct_weight'] * result['pct_used'] * result['shap_mean'])
    
    twsa = result.groupby(['field']).agg({'tot_wtd_shap_abs': 'sum'}).reset_index()
    twsa.rename(columns = {'tot_wtd_shap_abs': 'field_tot_wtd_shap_abs'}, inplace = True)
    
    tpu = result.groupby(['field']).agg({'pct_used': 'max'}).reset_index()
    tpu.rename(columns = {'pct_used': 'field_max_pct_used'}, inplace = True)
    
    result.drop(columns = ['tot_wtd_shap_abs'], inplace = True)
    result = result.merge(twsa).merge(tpu)
    
    result = result.loc[result['field_max_pct_used'] > 0]
    result.drop(columns = ['field_max_pct_used'], inplace = True)
    
    if sort_by_shap == True:
        result = result.sort_values(by = ['field_tot_wtd_shap_abs','field','val'], ascending = False)
        
    result = result.merge(all_mono_df, how = 'left')
    
    #result = result.loc[result['field'].isin(all_excluded_fields) == False]
        
    return result

In [ ]:
def shap_summary_display_all_features(train, weight, sort_by_shap=False):
    global df0
    df0 = train[model_xgb.feature_names].copy().reset_index().drop(columns="index")
    df = xgb.DMatrix(df0)
    fc = model_xgb.predict(df, pred_contribs=True)
    
    global fc_df
    fc_df = pd.DataFrame(fc, columns=model_xgb.feature_names + ["bias"])
    del fc_df["bias"]
    fc_df["weight"] = train[weight]

    # Process each feature individually
    global result
    result = pd.DataFrame()
    for field in [col for col in fc_df.columns if col != "weight"]:
        display = shap_display(field, shap_ridge_table(field, 5), 5)
        result = pd.concat([result, display])

    result["tot_wtd_shap_abs"] = np.abs(result["pct_weight"] * result["pct_used"] * result["shap_mean"])
    
    twsa = result.groupby(["field"]).agg({"tot_wtd_shap_abs": "sum"}).reset_index()
    twsa.rename(columns={"tot_wtd_shap_abs": "field_tot_wtd_shap_abs"}, inplace=True)
    
    tpu = result.groupby(["field"]).agg({"pct_used": "max"}).reset_index()
    tpu.rename(columns={"pct_used": "field_max_pct_used"}, inplace=True)
    
    result.drop(columns=["tot_wtd_shap_abs"], inplace=True)
    result = result.merge(twsa).merge(tpu)
    
    result = result.loc[result["field_max_pct_used"] > 0]
    result.drop(columns=["field_max_pct_used"], inplace=True)
    
    if sort_by_shap:
        result = result.sort_values(by=["field_tot_wtd_shap_abs", "field", "val"], ascending=False)
        
    result = result.merge(all_mono_df, how="left")
    result = result.loc[~result["field"].isin(all_excluded_fields)]
        
    return result

In [ ]:
def shap_fc_df_setup(data,weight_field):
    # get contribs and weight
    df0 = data[(model_xgb.feature_names)].copy().reset_index().drop(columns = 'index')    
    df = xgb.DMatrix(df0)
    fc = model_xgb.predict(df, pred_contribs=True)
    fc_df = pd.DataFrame(fc, columns = model_xgb.feature_names + ['bias'])
    del fc_df['bias']
    fc_df['weight'] = data.reset_index()[weight_field]
    
    return fc_df
    
def shap_cont_range_plot(fc_data,data,feature,weight_field,shap_round_level = 3,feature_round_to = 1,min_ntile = 0,max_ntile = 0, filter_used_only = False):
    global cf_df
    cf_df = data.reset_index()[([feature,weight_field])].copy()
    cf_df.rename(columns = {weight_field: 'weight'}, inplace = True)
    cf_df['contrib'] = fc_data[feature]

    # rounding
    if feature_round_to != 0: # 0 denotes no rounding
        cf_df[feature] = round(cf_df[feature]/feature_round_to,0)*feature_round_to
    cf_df['contrib'] = round(cf_df['contrib'],shap_round_level)
    
    # filter
    if filter_used_only == True:
        cf_df = cf_df.loc[cf_df['contrib'].fillna(0) != 0]

    # aggregate rounded data
    cf_df2 = cf_df.groupby([feature,'contrib']).agg({'weight': 'sum'}).reset_index()

    # create ntiles
    cf_df2['f_cumsum'] = cf_df2.groupby([feature]).weight.cumsum()

    f_weight = cf_df2.groupby([feature]).agg({'weight': 'sum'}).reset_index()
    f_weight.rename(columns = {'weight': 'f_weight'}, inplace = True)

    cf_df2 = cf_df2.merge(f_weight)

    cf_df2['ntile'] = round(cf_df2['f_cumsum']/cf_df2['f_weight'],2)*100
    
    cf_df2 = cf_df2.loc[cf_df2['ntile'].isna() == False]
    
    cf_df2['ntile'] = cf_df2['ntile'].astype('int')

    cf_df2['sp'] = cf_df2['contrib'] * cf_df2['weight']

    cf_df3 = cf_df2.groupby([feature,'ntile']).agg({'sp': 'sum', 'weight': 'sum'}).reset_index()
    cf_df3['SHAP'] = cf_df3['sp'] / cf_df3['weight']
    del cf_df3['sp'], cf_df3['weight']

    # set up SHAP data for plotting
    unique_feat_levels = cf_df3[feature].drop_duplicates().to_list()
    ntiles = [i for i in range(101)]

    u_df = pd.DataFrame()
    u_df[feature] = unique_feat_levels
    u_df['key'] = 0

    n_df = pd.DataFrame()
    n_df['ntile'] = ntiles
    n_df['key'] = 0

    df_levels = u_df.merge(n_df)
    del df_levels['key']

    cf_df3['ntile']
    cf_df3.sort_values(by = [feature,'ntile'], inplace = True)
    df_levels.sort_values(by = [feature,'ntile'], inplace = True)

    df = pd.DataFrame()
    for i in unique_feat_levels:
        a = df_levels.loc[df_levels[feature] == i].copy()
        b = cf_df3.loc[cf_df3[feature] == i].copy()

        # fill upwards
        a['ntile'] = a['ntile'].astype('int32')
        b['ntile'] = b['ntile'].astype('int32')
        c = pd.merge_asof(a, b ,on = 'ntile')

        # fill downwards
        lowest_contrib = c['SHAP'].min()
        c['SHAP'].fillna(lowest_contrib, inplace = True)

        df = pd.concat([df,c])

    df.rename(columns = {feature + '_x': feature}, inplace = True)
    del df[feature + '_y']

    df = df.loc[(df['ntile'] >= min_ntile) & (df['ntile'] <= max_ntile)]
    
    # distribution data
    a = cf_df.groupby([feature]).agg({'weight': 'sum'}).reset_index()
    b = cf_df.loc[cf_df['contrib'].fillna(0) != 0].groupby([feature]).agg({'weight': 'sum'}).reset_index()
    b.rename(columns = {'weight': 'used_weight'}, inplace = True)

    c = a.merge(b, how = 'left')
    c[feature] = round(c[feature],4)
    
    # plot
    g = sns.jointplot(x = df[feature], y = df['SHAP'], c = df['ntile'], height = 12, joint_kws={"color":None, 'cmap':'vlag'})
    
    g.fig.colorbar(g.ax_joint.collections[0], ax=[g.ax_joint, g.ax_marg_y, g.ax_marg_x], use_gridspec=True, orientation='vertical', shrink = .80, anchor = (0,0), label = 'SHAP Percentile', pad = -.15)
    
    g.fig.set_figwidth(12)
    g.fig.set_figheight(8)
    
    g.ax_marg_x.remove()
    g.ax_marg_y.remove()
    
    g.fig.suptitle('Continuous Feature SHAP Spread Plot; ' + feature, y = .9)
    
    plt.show()
    
    fig, ax = plt.subplots(figsize = (12,2))
    bar1 = sns.barplot(data = c, x = c[feature], y = c['weight'], color = 'grey', alpha = .3, ax = ax)
    bar2 = sns.barplot(data = c, x = c[feature], y = c['used_weight'], color = 'orange', alpha = .2, ax = ax)
    bar1.set(xlabel = None, ylabel = None)

In [ ]:
def shap_scatter_p1_p99(data,var,color_by):
    global a, b
    a = data.groupby([var]).agg({'weight': 'sum'}).reset_index()
    a['cumsum'] = a['weight'].cumsum()
    a['ntile'] = round(a['cumsum']/a[len(a)-1:len(a)]['cumsum'].values,3)
    xmax = a.loc[a['ntile'] >= .99][0:1][var].values

    if a.loc[a['ntile'] <= .01][::][var].shape[0] == 0:
        xmin = a[0:1][var].values
    else:
        b = a.loc[a['ntile'] <= .01][::][var].copy()
        xmin = b[len(b)-1:len(b)].values

    x = shap_values[:, var]
    shap.plots.scatter(x
                       , color=shap_values[:,color_by]
                       , xmin=xmin
                       , xmax=xmax
                      )

In [3]:
def trim_eda_table(path):
    global trim_eda, trim_blank_list, trim_blank_dtyp
    #path = path_prefix + 'marcusdeckert/Analytics/Symbols/2022/Related Analysis/Trim EDA/'
    #path = path_prefix + '/carfax_sym/Other Support/'

    trim_eda = pd.read_excel(path + '2024.03.11 VC Trim EDA Notes.xlsx', sheet_name = 'trim eda')

    
    
    trim_eda = trim_eda.loc[trim_eda['veh_type'] != '-'].copy()
    trim_eda.rename(columns = {'Unnamed: 2': 'category'}, inplace = True)
    trim_eda = trim_eda[0:36].copy() # for VAN COMP run change from 37 to 36
    
    #print(trim_eda)

    #print(trim_eda.head())
    
    trim_eda.fillna(method='ffill', inplace = True)

    trim_eda = trim_eda.loc[(trim_eda['veh_type'] == veh_type) & (trim_eda['cov'] == cov)]

    trim_eda.drop(columns = ['veh_type','cov'], inplace = True)
    trim_eda.set_index('category', inplace = True)

    trim_eda = trim_eda.T
    
    
    trim_eda.reset_index(inplace = True)
    trim_eda.rename(columns = {'index': 'field'}, inplace = True)

    df_dtyp = pd.DataFrame(data.dtypes)
    df_dtyp.reset_index(inplace = True)
    df_dtyp.rename(columns = {'index': 'field', 0: 'dtype'}, inplace = True)

    
    trim_eda = trim_eda.merge(df_dtyp)

    #print(trim_eda.head())
    #print(trim_eda['mono'] == 'x')
    mono_indices = np.where(trim_eda.columns == 'mono')[0]
    first_mono_col = trim_eda.iloc[:,mono_indices[0] ]
    print(first_mono_col)

    trim_blank_list = trim_eda.loc[first_mono_col == 'x']['field'].to_list()


    
    trim_dtyp = trim_eda.loc[first_mono_col  == 'x']['dtype'].to_list()
    trim_blank_dtyp = []
    for i in trim_dtyp:
        if i == 'bool':
            trim_blank_dtyp.append('bool')
        else:
            trim_blank_dtyp.append('num')
            
def options_mono_table(path):
    global options_mono_df, options_blank_list
    
    #path = path_prefix + 'marcusdeckert/Analytics/Symbols/2022/Related Analysis/Monotonicity Settings/'
   
    options_mono_df = pd.read_excel(path + '2024.03.08 Monotonicity Settings.xlsx', sheet_name = cov)
    options_mono_df = options_mono_df.loc[options_mono_df['vc_Vehicle_Type_M'] == veh_type]
    options_mono_df = options_mono_df[(['field','mono'])]
    options_mono_df = options_mono_df.loc[options_mono_df['field'].isin(data.columns)].reset_index().drop(columns = 'index')

    mono_indices = np.where(options_mono_df.columns == 'mono')[0]
    first_mono_col = options_mono_df.iloc[:,mono_indices[0] ]
    
    options_blank_list = options_mono_df.loc[ first_mono_col == 'x']['field'].to_list()
    
def get_mono_list():
    global all_mono_df, mono_list
    all_mono_df = pd.DataFrame()
    all_mono_df['field'] = features

    trim_mono = trim_eda[(['field','mono'])].copy()
    trim_mono.rename(columns = {'mono': 'trim_mono'}, inplace = True)

    all_mono_df = all_mono_df.merge(trim_mono, how = 'left')

    options_mono = options_mono_df.copy()
    options_mono.rename(columns = {'mono': 'options_mono'}, inplace = True)

    all_mono_df = all_mono_df.merge(options_mono, how = 'left')

    all_mono_df['mono'] = all_mono_df['trim_mono'].fillna(all_mono_df['options_mono'])

    all_mono_df = all_mono_df[(['field','mono'])]

    all_mono_df['mono'] = np.where(all_mono_df['mono'] == 'x', 0, all_mono_df['mono'])
    
    #print(all_mono_df)

    all_mono_df = all_mono_df.dropna()
    all_mono_df['mono'] = all_mono_df['mono'].astype('int8')

    mono_list = all_mono_df['mono'].to_list()

In [ ]:
def shagg_fn(fc_data):
    global shagg, shagg_num
    fc_df2 = fc_data.copy()

    cols = [i for i in fc_df2.columns if i != 'weight']
    for i in cols:
        fc_df2[i] = np.abs(fc_df2[i]) * fc_df2['weight']

    shagg = fc_df2.agg({i: 'sum' for i in cols}).reset_index()
    shagg.rename(columns = {'index': 'field', 0: 'total_shap'}, inplace = True)

    shagg = shagg.sort_values(by = 'total_shap', ascending = False)

    shagg_num = shagg.merge(trim_eda[(['field','dtype'])])
    shagg_num = shagg_num.loc[shagg_num['dtype'] != 'bool']
    shagg_num = shagg_num.loc[shagg_num['total_shap'] > 0]
    
    shagg_num['cum_shap_abs'] = shagg_num['total_shap'].cumsum()
    shagg_num['shap_abs_pct'] = shagg_num['total_shap']/shagg_num['total_shap'].sum()
    shagg_num['cum_shap_abs_pct'] = shagg_num['cum_shap_abs']/shagg_num['cum_shap_abs'].max()
    
    shagg_num.drop(columns = ['cum_shap_abs'], inplace = True)
    
    all_excluded_fields = trim_blank_list + options_blank_list + additional_blank_list
    shagg_num = shagg_num.loc[shagg_num['field'].isin(all_excluded_fields) == False]

In [ ]:
def shap_aggregate_fn(fc_data):
    global shagg, shagg_num, shagg2
    fc_df2 = fc_data.copy()

    cols = [i for i in fc_df2.columns if i != 'weight']
    for i in cols:
        fc_df2[i] = np.abs(fc_df2[i]) * fc_df2['weight']

    shagg = fc_df2.agg({i: 'sum' for i in cols}).reset_index()
    shagg.rename(columns = {'index': 'field', 0: 'total_shap'}, inplace = True)

    shagg = shagg.sort_values(by = 'total_shap', ascending = False)
    shagg2 = shagg.loc[shagg['total_shap'] > 0]
    shagg_num = shagg

    #shagg_num = shagg.merge(trim_eda[(['field','dtype'])])
    #shagg_num = shagg_num.loc[shagg_num['dtype'] != 'bool']
    shagg_num = shagg_num.loc[shagg_num['total_shap'] > 0]
    
    shagg_num['cum_shap_abs'] = shagg_num['total_shap'].cumsum()
    shagg_num['shap_abs_pct'] = shagg_num['total_shap']/shagg_num['total_shap'].sum()
    shagg_num['cum_shap_abs_pct'] = shagg_num['cum_shap_abs']/shagg_num['cum_shap_abs'].max()
    
    # shagg_num.drop(columns = ['cum_shap_abs'], inplace = True)
    
    # all_excluded_fields = trim_blank_list + options_blank_list + additional_blank_list
   # shagg_num = shagg_num.loc[shagg_num['field'].isin(all_excluded_fields) == False]